# ITCS 6169/8169 — Computer Vision
## Assignment 3: One-Shot Image Classification with Foundation Models

**Goal:** the highest possible test accuracy on the 16-class dataset using **exactly one labeled training image per class (16 images in total)**.

**Report deadline:** November 24, 2026, 11:59 PM ET. The report is at most **2 pages** and must contain (1) the best test accuracy plus a GitHub link, (2) your foundation-model / training recipe (**weighted 3x**), and (3) a conclusion.

### The one-shot rule, and how this template enforces it
| Allowed | Not allowed |
|---|---|
| Exactly 1 labeled image per class (16 total) | Any labels from the other training images |
| Pretrained foundation models (CLIP, SigLIP, DINOv2, ...) | Training or tuning on additional labeled images |
| Class names / text prompts | Using labeled held-out images (a "validation set") to tune |
| Augmenting your 16 images | Training on test labels |
| Test-time adaptation using **unlabeled** test images | |

`build_support_set()` is the only place labels are read from the training folder, and it asserts that exactly one image per class is returned. **Unlabeled** training images are off by default (`USE_UNLABELED_TRAIN = False`); the assignment text doesn't say whether they are permitted, so ask your instructor before turning that on.

Because there is no labeled validation data in a one-shot setting, any choice you make by looking at test accuracy (backbone, `ALPHA`, adapter settings, ...) is tuning on the test set. Keep your search small, decide as much as you can from priors and from the multi-seed averages below, and say in the report what you tuned on.

## 1. Setup

In [1]:
# Run once (Colab / fresh environment). SigLIP's tokenizer needs sentencepiece + protobuf.
%pip install -q transformers sentencepiece protobuf


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: /home/ward/MainDir/temp/CNN2/CNNvenv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 2.9 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: /home/ward/MainDir/temp/CNN2/CNNvenv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Core imports
from pathlib import Path
import random, time, math, gc, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torchvision import datasets
from IPython.display import display

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.14.1+cu130
CUDA available: True


In [4]:
# Optional Google Drive mount for Colab
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except ImportError:
    print("Not running in Google Colab.")

Not running in Google Colab.


## 2. Working directory

Expected dataset structure (the same 2,400-image / 16-class dataset as Assignments 1 and 2):

```text
data/
  train/
    class_1/
    class_2/
  test/
    class_1/
    class_2/
```

In [5]:
# Change this path to your project directory.
# PROJECT_ROOT = Path('/content/gdrive/MyDrive/ITCS_6169_8169/assignment3')
PROJECT_ROOT = Path('.')
DATA_ROOT = PROJECT_ROOT / 'data'
TRAIN_ROOT = DATA_ROOT / 'train'
TEST_ROOT = DATA_ROOT / 'test'

print("Project root:", PROJECT_ROOT.resolve())
print("Training data:", TRAIN_ROOT)
print("Testing data:", TEST_ROOT)
assert TRAIN_ROOT.exists(), "Training directory not found."
assert TEST_ROOT.exists(), "Testing directory not found."

Project root: /home/ward/MainDir/ComputerVision/FoundationModel
Training data: data/train
Testing data: data/test


AssertionError: Training directory not found.

In [ ]:
SEED = 0

def set_random_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_random_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

## 3. Configuration

Every knob for the experiment is here. The defaults are reasonable starting points, not tuned values.

In [ ]:
# (kind, Hugging Face model id). kind selects how features are extracted.
BACKBONES = {
    "clip_b16":      ("clip",   "openai/clip-vit-base-patch16"),
    "clip_l14":      ("clip",   "openai/clip-vit-large-patch14"),
    "siglip_b16":    ("siglip", "google/siglip-base-patch16-224"),
    "siglip_so400m": ("siglip", "google/siglip-so400m-patch14-384"),
    "dinov2_b":      ("dinov2", "facebook/dinov2-base"),     # image-only: no text prompts
}
BACKBONE = "clip_l14"

# ---- one-shot support set ---------------------------------------------------
SUPPORT_SEED = 0              # which single image per class is the labeled sample
SEEDS = [0, 1, 2, 3, 4]       # support sets to average over (reports robustness to the choice)
N_AUG_VIEWS = 32              # augmented views per support image (view 0 is the original)

# ---- recipe ------------------------------------------------------------------
ALPHA = 0.5                   # weight of text prototype vs image prototype in the fusion
USE_TTA = True                # test-time augmentation (flip + center crops, averaged)
ADAPTER_CFG = dict(steps=100, lr=2e-4, lam=10.0, scale=100.0, label_smoothing=0.1)
REFINE_CFG  = dict(iters=10, temp=0.03, gamma=0.5, balance=True)   # transductive refinement

# ---- unlabeled data -----------------------------------------------------------
# Test images are always used unlabeled for test-time adaptation (stage 7).
# Unlabeled TRAIN images are OFF by default: check with your instructor before enabling.
USE_UNLABELED_TRAIN = False

# Stage whose head is exported at the end: "adapter" (stage 6) or "refined" (stage 7)
FINAL_HEAD = "refined"

## 4. Data and the one-shot support set

Images are loaded as RGB PIL images (foundation models expect color input, and each model's own processor handles resizing and normalization). The folder is only used to look up **one** image per class; no other training labels are read.

In [ ]:
train_folder = datasets.ImageFolder(TRAIN_ROOT)   # PIL images, no transform
test_folder = datasets.ImageFolder(TEST_ROOT)
class_names = train_folder.classes
num_classes = len(class_names)
assert test_folder.class_to_idx == train_folder.class_to_idx, "Train and test class mappings do not match."

test_paths = [p for p, _ in test_folder.samples]
test_labels = torch.tensor(test_folder.targets)

print("Classes:", class_names)
print(f"Training images on disk: {len(train_folder)} (only {num_classes} will be used with labels)")
print(f"Test images: {len(test_folder)}")


def select_support_indices(folder, seed):
    """Pick one random image per class. This is the ONLY use of training labels."""
    rng = np.random.RandomState(seed)
    targets = np.asarray(folder.targets)
    return [int(rng.choice(np.where(targets == c)[0])) for c in range(len(folder.classes))]


def build_support_set(folder, seed):
    """Return (images, labels, file paths, indices) for exactly one image per class, ordered by class id."""
    idx = select_support_indices(folder, seed)
    images = [folder[i][0] for i in idx]
    labels = [folder.targets[i] for i in idx]
    files = [folder.samples[i][0] for i in idx]
    # Hard one-shot guard: 16 images, all distinct, exactly one per class
    assert len(idx) == len(set(idx)) == len(folder.classes), "Support set must have one image per class."
    assert labels == list(range(len(folder.classes))), "Support set must contain each class exactly once."
    return images, torch.tensor(labels), files, idx


# Unlabeled pool (paths only; labels are never read). Disabled by default.
unlabeled_paths = [p for p, _ in train_folder.samples] if USE_UNLABELED_TRAIN else []
print("Unlabeled training images used:", len(unlabeled_paths))

In [ ]:
def show_support_set(seed):
    images, labels, files, _ = build_support_set(train_folder, seed)
    cols = 4
    rows = math.ceil(len(images) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    for ax in np.asarray(axes).reshape(-1):
        ax.axis("off")
    for ax, img, lab in zip(np.asarray(axes).reshape(-1), images, labels):
        ax.imshow(img)
        ax.set_title(class_names[lab.item()])
    plt.suptitle(f"The {len(images)} labeled training images (support seed {seed})")
    plt.tight_layout()
    plt.show()

show_support_set(SUPPORT_SEED)

test_counts = pd.Series(test_folder.targets).value_counts().sort_index()
display(pd.DataFrame({"Class": class_names, "Labeled train (used)": 1, "Test": test_counts.values}))

## 5. Foundation-model utilities

The cell below writes `oneshot_lib.py` next to the notebook. It holds the feature extractor, augmentation, prototype/adapter code and test-time adaptation, so that `inference.py` in your GitHub repo can import exactly the same code. (If you edit it, run `importlib.reload(oneshot_lib)` or restart the kernel.)

In [ ]:
%%writefile oneshot_lib.py
"""Utilities for one-shot classification with foundation models (CLIP / SigLIP / DINOv2)."""
import re
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms as T

# ----------------------------------------------------------------------------
# Text prompts
# ----------------------------------------------------------------------------
PROMPT_TEMPLATES = [
    "a photo of a {}.",
    "a photo of the {}.",
    "a picture of a {}.",
    "a low resolution photo of a {}.",
    "a {} scene.",
    "an image showing a {}.",
]

# Human-readable phrasing for class folder names. Edit to match your class list.
DEFAULT_DESCRIPTIONS = {
    "Bedroom": "bedroom",
    "Coast": "coast with ocean and beach",
    "Flower": "flower",
    "Forest": "forest",
    "Highway": "highway road",
    "Industrial": "industrial building site",
    "InsideCity": "downtown city street",
    "Kitchen": "kitchen",
    "LivingRoom": "living room",
    "Mountain": "mountain landscape",
    "Office": "office interior",
    "OpenCountry": "open countryside field",
    "Store": "store interior with shelves",
    "Street": "city street",
    "Suburb": "suburban neighborhood with houses",
    "TallBuilding": "skyscraper",
}


def class_to_text(name, overrides=None):
    overrides = DEFAULT_DESCRIPTIONS if overrides is None else overrides
    if name in overrides:
        return overrides[name]
    return re.sub(r"(?<!^)(?=[A-Z])", " ", name).replace("_", " ").lower()


def _load(x):
    if isinstance(x, (str, Path)):
        return Image.open(x).convert("RGB")
    return x.convert("RGB") if x.mode != "RGB" else x


def _as_tensor(x):
    if torch.is_tensor(x):
        return x
    for attr in ("pooler_output", "image_embeds", "text_embeds"):
        v = getattr(x, attr, None)
        if v is not None:
            return v
    return x[0]


# ----------------------------------------------------------------------------
# Feature extractor
# ----------------------------------------------------------------------------
class FoundationModel:
    """Frozen backbone that returns L2-normalized image (and text) embeddings."""

    def __init__(self, kind, hf_name, device):
        from transformers import AutoModel, AutoProcessor, AutoTokenizer
        self.kind, self.hf_name, self.device = kind, hf_name, device
        model = AutoModel.from_pretrained(hf_name)
        if device.type == "cuda":
            model = model.half()
        self.model = model.to(device).eval()
        self.dtype = next(self.model.parameters()).dtype
        self.processor = AutoProcessor.from_pretrained(hf_name)
        self.tokenizer = None if kind == "dinov2" else AutoTokenizer.from_pretrained(hf_name)

    @property
    def has_text(self):
        return self.tokenizer is not None

    @torch.no_grad()
    def encode_images(self, images, batch_size=64):
        feats = []
        for i in range(0, len(images), batch_size):
            batch = [_load(x) for x in images[i:i + batch_size]]
            px = self.processor(images=batch, return_tensors="pt")["pixel_values"]
            px = px.to(self.device, dtype=self.dtype)
            if self.kind == "dinov2":
                h = self.model(pixel_values=px).last_hidden_state
                f = torch.cat([h[:, 0], h[:, 1:].mean(dim=1)], dim=-1)   # CLS + mean patch token
            else:
                f = _as_tensor(self.model.get_image_features(pixel_values=px))
            feats.append(F.normalize(f.float(), dim=-1).cpu())
        return torch.cat(feats)

    @torch.no_grad()
    def encode_texts(self, texts, batch_size=64):
        assert self.has_text, f"{self.kind} has no text encoder."
        feats = []
        for i in range(0, len(texts), batch_size):
            chunk = texts[i:i + batch_size]
            if self.kind == "siglip":
                tok = self.tokenizer(chunk, padding="max_length", max_length=64,
                                     truncation=True, return_tensors="pt")
                f = self.model.get_text_features(input_ids=tok["input_ids"].to(self.device))
            else:
                tok = self.tokenizer(chunk, padding=True, max_length=77,
                                     truncation=True, return_tensors="pt")
                f = self.model.get_text_features(
                    input_ids=tok["input_ids"].to(self.device),
                    attention_mask=tok["attention_mask"].to(self.device))
            feats.append(F.normalize(_as_tensor(f).float(), dim=-1).cpu())
        return torch.cat(feats)


def text_prototypes(fm, class_names, templates=PROMPT_TEMPLATES, overrides=None):
    """Zero-shot classifier: mean text embedding over prompt templates, one row per class."""
    protos = []
    for name in class_names:
        phrase = class_to_text(name, overrides)
        feats = fm.encode_texts([t.format(phrase) for t in templates])
        protos.append(F.normalize(feats.mean(0), dim=0))
    return torch.stack(protos)


# ----------------------------------------------------------------------------
# Augmentation: more views of the SAME single labeled image
# ----------------------------------------------------------------------------
_AUG = T.Compose([
    T.RandomResizedCrop(256, scale=(0.4, 1.0), ratio=(0.75, 1.333)),
    T.RandomHorizontalFlip(),
    T.RandomApply([T.ColorJitter(0.3, 0.3, 0.2, 0.02)], p=0.8),
    T.RandomApply([T.GaussianBlur(9, sigma=(0.1, 1.5))], p=0.3),
    T.RandomGrayscale(p=0.1),
])


def make_views(image, n_views, seed=0):
    """Return [original] + (n_views - 1) random augmentations (deterministic for a given seed)."""
    state = torch.get_rng_state()
    torch.manual_seed(seed)
    views = [image] + [_AUG(image) for _ in range(n_views - 1)]
    torch.set_rng_state(state)
    return views


def tta_views(image):
    """Deterministic test-time views: original, flip, and two center crops (each also flipped)."""
    w, h = image.size
    views = [image, T.functional.hflip(image)]
    for frac in (0.9, 0.75):
        cw, ch = int(w * frac), int(h * frac)
        left, top = (w - cw) // 2, (h - ch) // 2
        crop = image.crop((left, top, left + cw, top + ch))
        views += [crop, T.functional.hflip(crop)]
    return views


def encode_images_tta(fm, images, views_fn=tta_views, chunk=32):
    """Average the embeddings of several views of every image."""
    out = []
    for i in range(0, len(images), chunk):
        all_views, counts = [], []
        for x in images[i:i + chunk]:
            v = views_fn(_load(x))
            all_views += v
            counts.append(len(v))
        feats = fm.encode_images(all_views)
        j = 0
        for c in counts:
            out.append(F.normalize(feats[j:j + c].mean(0), dim=0))
            j += c
    return torch.stack(out)


def support_features(fm, support_images, n_views, seed=0):
    """Embeddings of augmented views of each support image -> [num_classes, n_views, D]."""
    feats = [fm.encode_images(make_views(im, n_views, seed + k)) for k, im in enumerate(support_images)]
    return torch.stack(feats)


def prototypes_from_views(view_feats):
    return F.normalize(view_feats.mean(dim=1), dim=-1)


# ----------------------------------------------------------------------------
# Classifier construction
# ----------------------------------------------------------------------------
def fuse(text_proto, image_proto, alpha=0.5):
    """Blend the zero-shot text prototype with the one-shot image prototype."""
    return F.normalize(alpha * text_proto + (1 - alpha) * image_proto, dim=-1)


def train_adapter(view_feats, init_W, steps=100, lr=2e-4, lam=10.0, scale=100.0,
                  label_smoothing=0.1, seed=0, device="cpu"):
    """
    Train a linear head on augmented views of the 16 support images.
    It starts from `init_W` (e.g. the fused prototypes) and an L2 term keeps it close to that
    initialisation, so the head can't simply memorise 16 images.
    """
    torch.manual_seed(seed)
    C, V, D = view_feats.shape
    X = view_feats.reshape(C * V, D).to(device)
    y = torch.arange(C).repeat_interleave(V).to(device)
    W0 = init_W.clone().to(device)
    W = torch.nn.Parameter(W0.clone())
    opt = torch.optim.AdamW([W], lr=lr, weight_decay=0.0)
    for _ in range(steps):
        logits = scale * X @ F.normalize(W, dim=-1).T
        loss = F.cross_entropy(logits, y, label_smoothing=label_smoothing)
        loss = loss + lam * ((W - W0) ** 2).sum(-1).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    return F.normalize(W.detach(), dim=-1).cpu()


# ----------------------------------------------------------------------------
# Transductive test-time adaptation (uses UNLABELED features only)
# ----------------------------------------------------------------------------
@torch.no_grad()
def sinkhorn(logits, n_iter=3):
    """Soft assignments with equal total mass per class (assumes a roughly balanced test set)."""
    logits = logits.double()
    Q = torch.exp(logits - logits.max())
    for _ in range(n_iter):
        Q = Q / Q.sum(0, keepdim=True)
        Q = Q / Q.sum(1, keepdim=True)
    return Q.float()


@torch.no_grad()
def transductive_refine(X, W0, iters=10, temp=0.03, gamma=0.5, balance=True):
    """Soft k-means in embedding space, anchored to the one-shot prototypes W0."""
    W = W0.clone()
    for _ in range(iters):
        logits = X @ W.T / temp
        Q = sinkhorn(logits) if balance else torch.softmax(logits, dim=1)
        centers = F.normalize(Q.T @ X, dim=-1)
        W = F.normalize((1 - gamma) * W0 + gamma * centers, dim=-1)
    return W


def accuracy(logits, labels):
    return (logits.argmax(dim=1) == labels).float().mean().item() * 100

In [ ]:
import importlib, oneshot_lib
importlib.reload(oneshot_lib)
from oneshot_lib import (
    FoundationModel, PROMPT_TEMPLATES, text_prototypes, encode_images_tta, support_features,
    prototypes_from_views, fuse, train_adapter, transductive_refine, accuracy, tta_views
)

## 6. The one-shot recipe, stage by stage

Each stage adds one idea, so the results table doubles as an **ablation study** for the report:

| Stage | Idea | Labeled data used |
|---|---|---|
| 1. Zero-shot | Match the image to text prompts such as "a photo of a kitchen." | none (class names only) |
| 2. One-shot prototype | Match to the single raw support image of each class | 16 images |
| 3. + augmented views | Average embeddings of many augmentations of each support image for a more robust prototype | same 16 images |
| 4. + fusion | Blend the text prototype and the image prototype (`ALPHA`) | same 16 images |
| 5. + adapter | Train a linear head on augmented support views, anchored to the fused prototypes | same 16 images |
| 6. + test-time augmentation | Average embeddings of several views of each test image | none |
| 7. + transductive refinement | Soft k-means on unlabeled test features, with class-balanced (Sinkhorn) assignments, anchored to the prototypes | none (unlabeled test images) |

Stage 7 assumes classes are roughly balanced in the unlabeled pool. State that assumption in the report.

In [ ]:
M1 = "1. Zero-shot (text prompts)"
M2 = "2. One-shot prototype (raw image)"
M3 = "3. + augmented support views"
M4 = "4. + text/image fusion"
M5 = "5. + adapter training"
M6 = "6. + test-time augmentation"
M7 = "7. + transductive refinement"
ALL_METHODS = [M1, M2, M3, M4, M5, M6, M7]


def run_pipeline(backbone_key, seeds=SEEDS):
    """Encode everything once with a frozen backbone, then evaluate all stages for each support set."""
    kind, hf_name = BACKBONES[backbone_key]
    t0 = time.time()
    fm = FoundationModel(kind, hf_name, device)
    text_proto = text_prototypes(fm, class_names) if fm.has_text else None

    # Backbone is frozen, so test features are computed once and reused for every support set.
    Xc = fm.encode_images(test_paths)
    Xt = encode_images_tta(fm, test_paths) if USE_TTA else Xc
    Xu = encode_images_tta(fm, unlabeled_paths) if unlabeled_paths else None
    pool = Xt if Xu is None else torch.cat([Xt, Xu])        # unlabeled features for stage 7
    print(f"[{backbone_key}] features ready in {time.time() - t0:.0f}s")

    per_seed, kept = {}, {}
    for seed in seeds:
        imgs, _, files, _ = build_support_set(train_folder, seed)
        view_feats = support_features(fm, imgs, N_AUG_VIEWS, seed)       # [C, V, D]
        sup_raw = F.normalize(view_feats[:, 0], dim=-1)                  # view 0 = original image
        sup_aug = prototypes_from_views(view_feats)
        fused = fuse(text_proto, sup_aug, ALPHA) if text_proto is not None else sup_aug
        W_ad = train_adapter(view_feats, fused, device=device, seed=seed, **ADAPTER_CFG)
        W_ref = transductive_refine(pool, W_ad, **REFINE_CFG)

        logits = {}
        if text_proto is not None:
            logits[M1] = Xc @ text_proto.T
        logits[M2] = Xc @ sup_raw.T
        logits[M3] = Xc @ sup_aug.T
        logits[M4] = Xc @ fused.T
        logits[M5] = Xc @ W_ad.T
        logits[M6] = Xt @ W_ad.T
        logits[M7] = Xt @ W_ref.T
        per_seed[seed] = {m: accuracy(l, test_labels) for m, l in logits.items()}
        kept[seed] = dict(logits=logits, W_adapter=W_ad, W_refined=W_ref, files=files)

    table = pd.DataFrame(per_seed).reindex(ALL_METHODS)
    table["mean"] = table[list(seeds)].mean(axis=1)
    table["std"] = table[list(seeds)].std(axis=1)
    return dict(fm=fm, table=table, kept=kept, backbone=backbone_key)

### 6.1 Run the pipeline

Columns are accuracies (%) for different randomly chosen support sets, plus mean and standard deviation. The standard deviation shows how much the result depends on **which** single image represents each class, which is usually the biggest source of variance in one-shot learning.

In [ ]:
result = run_pipeline(BACKBONE)
display(result["table"].round(2))

In [ ]:
table = result["table"]
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(table.index[::-1], table["mean"][::-1], xerr=table["std"][::-1], capsize=3)
ax.set_xlabel("Test accuracy (%), mean ± std over support sets")
ax.set_title(f"Ablation: {BACKBONE}")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Error analysis for the final configuration

Use this to write the "observations" part of the report: which classes get confused, and why (for example Street / InsideCity / TallBuilding, or Bedroom / LivingRoom).

In [ ]:
final_stage = M7 if FINAL_HEAD == "refined" else M6
kept = result["kept"][SUPPORT_SEED]
preds = kept["logits"][final_stage].argmax(dim=1).numpy()
labels_np = test_labels.numpy()
print(f"{final_stage} | support seed {SUPPORT_SEED} | test accuracy: {(preds == labels_np).mean() * 100:.2f}%")

cm = np.zeros((num_classes, num_classes), dtype=int)
for t, p in zip(labels_np, preds):
    cm[t, p] += 1

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(num_classes)); ax.set_xticklabels(class_names, rotation=90)
ax.set_yticks(range(num_classes)); ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for i in range(num_classes):
    for j in range(num_classes):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=7)
plt.colorbar(im)
plt.tight_layout()
plt.show()

per_class = pd.DataFrame({"Class": class_names, "Accuracy (%)": 100 * cm.diagonal() / cm.sum(axis=1)})
display(per_class.sort_values("Accuracy (%)").round(1))

## 8. Compare foundation models (optional)

The assignment asks you to **share your observations** about foundation models. This cell runs the same pipeline for every backbone, which downloads several models and can take a while. It is off by default.

In [ ]:
RUN_BACKBONE_SWEEP = False

if RUN_BACKBONE_SWEEP:
    rows = []
    for key in BACKBONES:
        r = result if key == BACKBONE else run_pipeline(key)
        t = r["table"]
        rows.append({
            "Backbone": key,
            "Zero-shot": t.loc[M1, "mean"],
            "One-shot prototype": t.loc[M2, "mean"],
            "+ fusion": t.loc[M4, "mean"],
            "+ adapter + TTA": t.loc[M6, "mean"],
            "+ transductive": t.loc[M7, "mean"],
        })
        if r is not result:
            del r
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    sweep_table = pd.DataFrame(rows).set_index("Backbone")
    display(sweep_table.round(2))
else:
    print("Set RUN_BACKBONE_SWEEP = True to compare all backbones.")

## 9. Experiment notes and report guidance

Record each experiment: what you changed, why, and what you learned. Remember that every row below is evaluated on the test set, so disclose that in the report.

| Experiment | What changed? | Why did you try it? | Mean test accuracy | What did you learn? |
|---|---|---|---:|---|
| Stage 1 | Zero-shot text prompts | Reference point with no labeled images | | |
| Stage 2 | One-shot image prototypes | How far do 16 images alone get us? | | |
| Stage 3 | Augmented support views | Is a single image a noisy prototype? | | |
| Stage 4 | Text and image fusion | Do the two modalities complement each other? | | |
| Stage 5 | Adapter training | Does anchored training beat plain prototypes? | | |
| Stage 6 | Test-time augmentation | Cheap robustness | | |
| Stage 7 | Transductive refinement | Can unlabeled test features fix prototype bias? | | |
| Backbone sweep | CLIP vs SigLIP vs DINOv2 | Which pretraining objective transfers best? | | |

### Report outline (max 2 pages)
1. **Maximum accuracy and code (equal weight):** the best test accuracy, a GitHub link, and a note that the repo contains the head weights, the 16 support images, inference code and reproduction steps. State which support seed you used and the mean ± std over seeds.
2. **Foundation model / training recipe (3x weight):** the backbone and why, how prompts and augmentations are built, fusion, the adapter, test-time adaptation, an ablation table with your observations (for example how much each stage adds, what hurts, and how sensitive results are to the support image), and your tuning policy.
3. **Conclusion (equal weight):** contributions, observations, and key takeaways. Discuss the balance between model size, data availability and compute, for example the accuracy gain from a larger backbone against its cost.

### Questions worth answering in your observations
- Zero-shot vs one-shot: which is stronger, and which classes does each get wrong?
- Does the choice of the single support image matter (look at the std across seeds)?
- Does text-image fusion beat either alone? Is the best `ALPHA` backbone-dependent?
- Does SigLIP beat CLIP here? Does a bigger model always help?
- Does stage 7 help? It leans on the class-balance assumption and on a decent starting point. Why?

## 10. Export for GitHub: head, support set, inference code

The "trained model" is the frozen pretrained backbone (downloaded from Hugging Face by name) plus the small learned classification head. The cell below saves the head and configuration, copies the 16 support images, and writes `inference.py`, `README.md` and `requirements.txt`.

In [ ]:
kept = result["kept"][SUPPORT_SEED]
W_final = kept["W_refined"] if FINAL_HEAD == "refined" else kept["W_adapter"]
kind, hf_name = BACKBONES[BACKBONE]

# Copy the 16 labeled images so the repo shows exactly what was used
support_dir = PROJECT_ROOT / "support_set"
support_dir.mkdir(exist_ok=True)
support_files = []
for cls, f in zip(class_names, kept["files"]):
    dst = support_dir / f"{cls}{Path(f).suffix}"
    shutil.copy(f, dst)
    support_files.append(str(dst.relative_to(PROJECT_ROOT)))

artifacts = dict(
    backbone_key=BACKBONE,
    backbone_kind=kind,
    hf_name=hf_name,
    class_names=list(class_names),
    W=W_final.cpu(),                          # [num_classes, D] normalized classification head
    tta=bool(USE_TTA),
    transductive=(FINAL_HEAD == "refined"),   # stage 7 is applied to the batch of test images
    refine_cfg=dict(REFINE_CFG),
    support_seed=int(SUPPORT_SEED),
    support_files=support_files,
    prompt_templates=list(PROMPT_TEMPLATES),
)
torch.save(artifacts, PROJECT_ROOT / "oneshot_artifacts.pt")
print("Saved oneshot_artifacts.pt | head shape:", tuple(W_final.shape))
print("Support images copied to:", support_dir)

In [ ]:
%%writefile inference.py
"""
One-shot scene classification inference.

Examples:
    python inference.py --image path/to/photo.jpg
    python inference.py --data data/test            # ImageFolder layout; prints accuracy
    python inference.py --data data/test --no-transductive
"""
import argparse
from pathlib import Path

import torch
import torch.nn.functional as F
from torchvision import datasets

from oneshot_lib import FoundationModel, encode_images_tta, transductive_refine, accuracy


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--artifacts", default="oneshot_artifacts.pt")
    ap.add_argument("--image", help="classify a single image file")
    ap.add_argument("--data", help="ImageFolder-style directory; reports accuracy")
    ap.add_argument("--no-transductive", action="store_true",
                    help="skip test-time refinement on the unlabeled batch")
    args = ap.parse_args()
    assert args.image or args.data, "Pass --image or --data."

    art = torch.load(args.artifacts, map_location="cpu", weights_only=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    fm = FoundationModel(art["backbone_kind"], art["hf_name"], device)
    W, names = art["W"], art["class_names"]

    def embed(paths):
        return encode_images_tta(fm, paths) if art["tta"] else fm.encode_images(paths)

    if args.image:
        feats = embed([args.image])
        probs = torch.softmax(100.0 * feats @ W.T, dim=1)[0]
        top = probs.topk(3)
        for p, i in zip(top.values, top.indices):
            print(f"{names[i]:>14s}  {p.item() * 100:5.1f}%")
        return

    folder = datasets.ImageFolder(args.data)
    assert folder.classes == names, "Class folders do not match the trained head."
    paths = [p for p, _ in folder.samples]
    labels = torch.tensor(folder.targets)
    feats = embed(paths)
    if art["transductive"] and not args.no_transductive:
        W = transductive_refine(feats, W, **art["refine_cfg"])
    print(f"Accuracy on {len(paths)} images: {accuracy(feats @ W.T, labels):.2f}%")


if __name__ == "__main__":
    main()

In [ ]:
(PROJECT_ROOT / "requirements.txt").write_text(
    "torch\ntorchvision\ntransformers\nsentencepiece\nprotobuf\npillow\nnumpy\npandas\nmatplotlib\n"
)

readme = f"""# One-shot scene classification (ITCS 6169/8169, Assignment 3)

One labeled training image per class (16 in total) plus the pretrained foundation model
`{hf_name}`. Final test accuracy: **FILL IN** (support seed {SUPPORT_SEED}).

## Contents
- `notebook.ipynb` - the full experiment (rename to match your file)
- `oneshot_lib.py` - feature extractor, augmentation, prototypes, adapter, test-time adaptation
- `inference.py` - run the trained head on an image or an ImageFolder directory
- `oneshot_artifacts.pt` - the trained head (`W`, {tuple(W_final.shape)}) and configuration
- `support_set/` - the 16 labeled training images that were used
- `requirements.txt`

## Reproduce
```bash
pip install -r requirements.txt
# put the dataset at data/train and data/test, then:
python inference.py --data data/test
python inference.py --image some_photo.jpg
```
The backbone weights are downloaded automatically from Hugging Face. To retrain the head,
run the notebook top to bottom (set `SUPPORT_SEED = {SUPPORT_SEED}`).

## Notes
- Only the 16 images in `support_set/` are used with labels.
- Unlabeled test images are used for test-time adaptation (stage 7) when `transductive` is true.
"""
(PROJECT_ROOT / "README.md").write_text(readme)
print("Wrote requirements.txt and README.md")

## 11. Final test evaluation

Report the result for the support set you committed to **before** looking at test scores, together with the mean ± std over seeds. Choosing the best-scoring seed by test accuracy would overstate how well the method really works.

In [ ]:
final_logits = result["kept"][SUPPORT_SEED]["logits"][final_stage]
print(f"Final test accuracy ({final_stage}, support seed {SUPPORT_SEED}): {accuracy(final_logits, test_labels):.2f}%")
t = result["table"].loc[final_stage]
print(f"Mean ± std over {len(SEEDS)} support sets: {t['mean']:.2f} ± {t['std']:.2f}%")

# Sanity check: the exported inference path should reproduce this number
# (run in a terminal:  python inference.py --data data/test)

## 12. AI-assisted development checkpoint

If you use an AI coding assistant, document the tools used, representative ways AI assisted, at least one incorrect or ineffective suggestion, how you verified or corrected it, and one important decision you made yourself. Verify tensor shapes, model names, assumptions, and code behavior before relying on generated code.

## 13. GitHub and submission checklist

```text
README.md
AI_USAGE.md
notebook.ipynb
oneshot_lib.py
inference.py
oneshot_artifacts.pt        # trained head + config
support_set/                # the 16 labeled images
requirements.txt
```

Before submitting the 2-page PDF (due November 24, 2026, 11:59 PM ET):
- The report states the best test accuracy and links the GitHub repo.
- A fresh clone can run `python inference.py --data data/test` and reproduce the number.
- Exactly 16 labeled training images were used, and the unlabeled-data policy is stated.
- The tuning policy is disclosed, including that the test set was used to compare variants.
- The recipe section explains each stage and your observations about the foundation models.